In [ ]:
RAG PDF Questions 

In [1]:
import os 
import numpy as np
import faiss
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

In [2]:
# load api Key 
load_dotenv()
client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

In [3]:
def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ''
    for page in reader.pages:
        text += page.extract_text() + '\n'
    return text    

In [4]:
def chunk_text(text, chunk_size=500 , overlap=100):
    chunks = []
    start = 0
    while start + len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks    

In [5]:
def get_embeddings(texts):
    response = client.embedding.create(
        model = 'text-embedding-3-small',
        input=texts
    )
    return np.array([d.embedding for d in response.data]).astype('float32')

In [6]:
def create_faiss_index(embedding):
    dimension = embedding.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embedding)
    return index

In [7]:
def search(query , index, chunks, k=3):
    query_embedding = get_embedding([query])
    distances, indices - index.search(query_embedding, k)
    return[chunks[i] for i in indices[0]]
    

In [8]:
def ask_questions(query, index, chunks):
    relevant_chunks = search(query, index, chunks)
    context = '\n\n'.join(relevant_chunks)

    prompt = f'''Answer the questions based only on the context below.

context:
{context}

Question:
{query}'''


    response = client.chat.completions.craete(
         model='gpt-4o-mini',
         messages=[
             {'role':'system','content':'you are a helpful assistant.'},
             {'role':'user','content':prompt}       
         ]
     )
    return response.choices[0].message.content

In [ ]:
# main execution 

pdf_text = read_pdf('../Data/Dr_Ashish_Chandiok_RAG_Resume.pdf')
chunks = chunk_text(pdf_text)
embeddings = get_embeddings(chunks)
index = create_faiss_index(embeddings)

print('RAG Ready!')

In [ ]:
while True:
    question = input("\nAsk a question (Type exit or stop):")

    if question.lower() in ['exit','stop']:
        break

    answer = ask_question(question, index , chunks)
    print("\nAnswer:", answer)
    